In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

In [2]:
# ==========================================
# 1. LOAD DATA & INITIAL CLEANING
# ==========================================
df = pd.read_csv('dataset_akt23.csv')

# Mengatasi duplikasi dan missing value standar
df = df.drop_duplicates()
df['IPK'] = df['IPK'].fillna(df['IPK'].median())
df['IPS_Sem6'] = df['IPS_Sem6'].fillna(df['IPS_Sem6'].median())
df['Jml_MK_Gagal'] = df['Jml_MK_Gagal'].fillna(0)
df['SKS_Lulus'] = df['SKS_Lulus'].fillna(df['SKS_Lulus'].median())
df['Jumlah_Bimbingan'] = df['Jumlah_Bimbingan'].fillna(0)
df['Status_Skripsi'] = df['Status_Skripsi'].fillna('Belum Ambil')

# Standarisasi nilai string agar rapi
df['Status_Skripsi'] = df['Status_Skripsi'].str.strip()

In [3]:
# ==========================================
# 2. RULE-BASED SYSTEM (Sinkronisasi Tahapan)
# ==========================================
# Tahapan yang diakui sebagai progres aktif (Skripsi merujuk pada tahap akhir/Sidang Komprehensif)
tahapan_aktif = ['Judul ACC', 'Seminar Proposal', 'Seminar Hasil', 'Sidang Komprehensif']

def tentukan_label_atasan(row):
    # Mahasiswa dinilai ONTIME jika memenuhi SEMUA kriteria berikut:
    # 1. IPK aman (>= 3.00)
    # 2. MK Gagal minim (<= 2)
    # 3. Jumlah Bimbingan aktif berjalan (> 5 kali bimbingan sebagai batas rasional data riil)
    # 4. Berada dalam tahapan skripsi aktif (Judul ACC, Sempro, Semhas, atau Sidang Komprehensif)
    
    if (row['IPK'] >= 3.00) and \
       (row['Jml_MK_Gagal'] <= 2) and \
       (row['Jumlah_Bimbingan'] > 5) and \
       (row['Status_Skripsi'] in tahapan_aktif):
        return 'Ontime'
    else:
        return 'Beresiko'

df['Target'] = df.apply(tentukan_label_atasan, axis=1)

In [4]:
# ==========================================
# 3. FEATURE ENGINEERING & TRANSFORMATION
# ==========================================
df['Rata_IP'] = (df['IPK'] + df['IPS_Sem6']) / 2
df['Progress_SKS'] = df['SKS_Lulus'] / 144
df['Tren_IP'] = df['IPS_Sem6'] - df['IPK']

# Transformasi Data Kategorikal (Status Skripsi)
le_status = LabelEncoder()
df['Status_Skripsi_Encoded'] = le_status.fit_transform(df['Status_Skripsi'])

le_target = LabelEncoder()
df['Target_Encoded'] = le_target.fit_transform(df['Target'])

# ==========================================
# 4. FEATURE SELECTION (Mencegah Data Leakage)
# ==========================================
# Mengeluarkan NIM, Nama, Tahun_Masuk, Bulan_Lulus, dan Target teks asli
fitur = ['IPK', 'IPS_Sem6', 'Jml_MK_Gagal', 'SKS_Lulus', 'Jumlah_Bimbingan', 
         'Status_Skripsi_Encoded', 'Rata_IP', 'Progress_SKS', 'Tren_IP']

X = df[fitur]
y = df['Target_Encoded']

In [5]:
# ==========================================
# 5. MODELING & EVALUASI
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Cek Akurasi Singkat
y_pred = model.predict(X_test)
print(f"Akurasi Model ML: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nLaporan Klasifikasi:\n", classification_report(y_test, y_pred, target_names=le_target.classes_))

Akurasi Model ML: 100.00%

Laporan Klasifikasi:
               precision    recall  f1-score   support

    Beresiko       1.00      1.00      1.00        31
      Ontime       1.00      1.00      1.00         9

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



In [6]:
# ==========================================
# 6. EXPORT ARTIFACT UNTUK DEPLOYMENT
# ==========================================
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('encoder_status.pkl', 'wb') as f:
    pickle.dump(le_status, f)

with open('encoder_target.pkl', 'wb') as f:
    pickle.dump(le_target, f)

print("Semua komponen AI berhasil disinkronkan dan disimpan!")

Semua komponen AI berhasil disinkronkan dan disimpan!
